# capit — train on Colab (Stage 3.3)

A thin launcher: all the logic lives in `train.py` in the repo. This notebook clones it, stages the data, and runs the CLI on the T4.

**Before you run:**
1. **Runtime → Change runtime type → T4 GPU.**
2. Locally: `python pipeline/scripts/make_train_zip.py` → builds `data/flickr8k_colab.zip` (Images + dataset_flickr8k.json + vocab.json).
3. Upload `flickr8k_colab.zip` to Google Drive at **`MyDrive/capit/`**.
4. **Push your latest code** — the notebook runs what is in git, not your local working tree.

Free-tier disconnects are expected: reconnect, re-run all cells, and `--resume auto` continues from the last checkpoint on Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# torch/torchvision are preinstalled on Colab; this pulls the small extras (nltk, ...).
# No -q: a failed install must be visible, not surface 3 hours later as ModuleNotFoundError.
!rm -rf /content/capit && git clone https://github.com/Bukunmi2108/capit.git /content/capit
!pip install -e /content/capit/pipeline
import capit  # fail fast if the install didn't take
print("capit installed")

In [ ]:
# Copy OFF the Drive mount to local disk, then unzip (never read images over Drive — slow).
# `&&` so unzip only runs if the copy succeeded (missing zip / unmounted Drive aborts here).
!cp /content/drive/MyDrive/capit/flickr8k_colab.zip /content/flickr8k_colab.zip && unzip -q -o /content/flickr8k_colab.zip -d /content/flickr8k
import os
n = len(os.listdir('/content/flickr8k/Images'))
assert n == 8091, f"expected 8091 images, got {n} — bad/partial zip or Drive copy failed"
print(n, "images staged")

In [ ]:
!python -m capit.train \
  --data-root /content/flickr8k \
  --vocab-path /content/flickr8k/vocab.json \
  --ckpt-dir /content/drive/MyDrive/capit/checkpoints \
  --resume auto

## If disconnected
Reconnect → re-run **all** cells (mount, install, data, train). `--resume auto` loads `latest.pt` from Drive and continues from the next epoch — checkpoints live on Drive, so a disconnect costs minutes, not the run. (train.py refuses to run if `--ckpt-dir` points at an unmounted Drive, so you can't silently lose checkpoints to ephemeral disk.)

**Exit gate (Stage 3.3):** `best.pt` on Drive with val BLEU-4 (greedy, nltk) ≥ ~14. Download `MyDrive/capit/checkpoints/best.pt` for Phase 4.